# NERA Risk Intelligence — 04. Temporal Sequence Modeling (LSTM/GRU)
### Learning Multi-Day Weather Curves (14-Day Cumulative Rain Evolution)

While tabular XGBoost observes aggregate summary statistics (`rainfall_72h`, `rainfall_7d`), the **Bidirectional LSTM** learns sequential trajectory dynamics:
- Sustained soil saturation accumulation
- Rapid convective spike patterns vs gentle continuous drizzle
- Antecedent vulnerability buildup prior to severe hazard trigger

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path

# PyTorch Model Definition
class TemporalDisruptionLSTM(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        lstm_out, _ = self.lstm(x)
        # Take last time-step representation
        last_step = lstm_out[:, -1, :]
        prob = self.fc(last_step)
        return prob

print("Temporal LSTM Model Architecture Initialized.")